# Colab Train Model

Notebook n?y t?i ?u cho **Google Colab** theo h??ng:
- t?i dataset t? Kaggle
- ??c CSV theo t?ng `chunk` ?? tr?nh tr?n RAM
- ch? gi? c?c c?t c?n thi?t
- sample theo file ?? v?n train ???c tr?n Colab free
- train Random Forest v? Autoencoder
- v? s? ?? v? xu?t artifacts

H?y ch?y t? tr?n xu?ng d??i. N?u s?a code, h?y `Runtime -> Restart session` r?i ch?y l?i t? ??u.


In [ ]:
!pip -q install kaggle numpy pandas scikit-learn tensorflow joblib dill lime matplotlib


## 1. Upload kaggle.json


In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    print('Upload kaggle.json from your machine')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('You must upload kaggle.json')
    if Path('kaggle.json').exists():
        shutil.move('kaggle.json', kaggle_json)

os.chmod(kaggle_json, 0o600)
print('Using Kaggle credentials at', kaggle_json)


## 2. C?u h?nh train

N?u Colab v?n thi?u RAM, gi?m ti?p:
- `FILE_SAMPLE_FRAC`
- `MAX_ROWS_PER_FILE`
- `CHUNK_SIZE`
- `AE_MAX_BENIGN_TRAIN_ROWS`


In [ ]:
DATASET_SLUG = 'remarezirafsanjani/cse-cic-ids2018-cleaned-and-match-cicflowmeter'
BENIGN_LABEL = 'Benign'
LABEL_COLUMN_OVERRIDE = ''
DOWNLOAD_DIR = '/content/kaggle_data'
OUTPUT_DIR = '/content/models_out'

FILE_SAMPLE_FRAC = 0.01
MAX_ROWS_PER_FILE = 5000
CHUNK_SIZE = 50000
RANDOM_STATE = 42
TEST_SIZE = 0.2
RF_ESTIMATORS = 250
AE_EPOCHS = 30
AE_BATCH_SIZE = 256
AE_MAX_BENIGN_TRAIN_ROWS = 25000
LIME_MAX_TRAIN_ROWS = 3000

print('DATASET_SLUG =', DATASET_SLUG)
print('BENIGN_LABEL =', BENIGN_LABEL)
print('DOWNLOAD_DIR =', DOWNLOAD_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('FILE_SAMPLE_FRAC =', FILE_SAMPLE_FRAC)
print('MAX_ROWS_PER_FILE =', MAX_ROWS_PER_FILE)
print('CHUNK_SIZE =', CHUNK_SIZE)
print('RANDOM_STATE =', RANDOM_STATE)


## 3. Import v? h?m ti?n ?ch


In [ ]:
import gc
import glob
import json
import os
import pickle
import re
import shutil
import subprocess
from pathlib import Path

import dill
import joblib
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lime import lime_tabular
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

APP_FEATURES = [
    'FlowDuration', 'BwdPacketLenMax', 'BwdPacketLenMin', 'BwdPacketLenMean', 'BwdPacketLenStd',
    'FlowIATMean', 'FlowIATStd', 'FlowIATMax', 'FlowIATMin', 'FwdIATTotal', 'FwdIATMean',
    'FwdIATStd', 'FwdIATMax', 'FwdIATMin', 'BwdIATTotal', 'BwdIATMean', 'BwdIATStd',
    'BwdIATMax', 'BwdIATMin', 'FwdPSHFlags', 'FwdPackets_s', 'MaxPacketLen', 'PacketLenMean',
    'PacketLenStd', 'PacketLenVar', 'FINFlagCount', 'SYNFlagCount', 'PSHFlagCount',
    'ACKFlagCount', 'URGFlagCount', 'AvgPacketSize', 'AvgBwdSegmentSize', 'InitWinBytesFwd',
    'InitWinBytesBwd', 'ActiveMin', 'IdleMean', 'IdleStd', 'IdleMax', 'IdleMin'
]

FEATURE_ALIASES = {
    'FlowDuration': ['Flow Duration'],
    'BwdPacketLenMax': ['Bwd Packet Length Max', 'BwdPacketLengthMax', 'Bwd Pkt Len Max'],
    'BwdPacketLenMin': ['Bwd Packet Length Min', 'BwdPacketLengthMin', 'Bwd Pkt Len Min'],
    'BwdPacketLenMean': ['Bwd Packet Length Mean', 'BwdPacketLengthMean', 'Bwd Pkt Len Mean'],
    'BwdPacketLenStd': ['Bwd Packet Length Std', 'BwdPacketLengthStd', 'Bwd Pkt Len Std'],
    'FlowIATMean': ['Flow IAT Mean'],
    'FlowIATStd': ['Flow IAT Std'],
    'FlowIATMax': ['Flow IAT Max'],
    'FlowIATMin': ['Flow IAT Min'],
    'FwdIATTotal': ['Fwd IAT Total', 'Fwd IAT Tot'],
    'FwdIATMean': ['Fwd IAT Mean'],
    'FwdIATStd': ['Fwd IAT Std'],
    'FwdIATMax': ['Fwd IAT Max'],
    'FwdIATMin': ['Fwd IAT Min'],
    'BwdIATTotal': ['Bwd IAT Total', 'Bwd IAT Tot'],
    'BwdIATMean': ['Bwd IAT Mean'],
    'BwdIATStd': ['Bwd IAT Std'],
    'BwdIATMax': ['Bwd IAT Max'],
    'BwdIATMin': ['Bwd IAT Min'],
    'FwdPSHFlags': ['Fwd PSH Flags'],
    'FwdPackets_s': ['Fwd Packets/s', 'FwdPackets/s', 'Fwd Pkts/s'],
    'MaxPacketLen': ['Max Packet Length', 'Packet Length Max', 'PacketLengthMax', 'Pkt Len Max'],
    'PacketLenMean': ['Packet Length Mean', 'PacketLengthMean', 'Pkt Len Mean'],
    'PacketLenStd': ['Packet Length Std', 'PacketLengthStd', 'Pkt Len Std'],
    'PacketLenVar': ['Packet Length Variance', 'PacketLengthVariance', 'Pkt Len Var'],
    'FINFlagCount': ['FIN Flag Count', 'FIN Flag Cnt'],
    'SYNFlagCount': ['SYN Flag Count', 'SYN Flag Cnt'],
    'PSHFlagCount': ['PSH Flag Count', 'PSH Flag Cnt'],
    'ACKFlagCount': ['ACK Flag Count', 'ACK Flag Cnt'],
    'URGFlagCount': ['URG Flag Count', 'URG Flag Cnt'],
    'AvgPacketSize': ['Average Packet Size', 'AveragePacketSize', 'Pkt Size Avg'],
    'AvgBwdSegmentSize': ['Avg Bwd Segment Size', 'Bwd Segment Size Avg', 'BwdSegmentSizeAvg', 'Bwd Seg Size Avg'],
    'InitWinBytesFwd': ['Init_Win_bytes_forward', 'Fwd Init Win Bytes', 'FWDInitWinBytes', 'FwdInitWinBytes', 'Init Fwd Win Byts'],
    'InitWinBytesBwd': ['Init_Win_bytes_backward', 'Bwd Init Win Bytes', 'BwdInitWinBytes', 'Init Bwd Win Byts'],
    'ActiveMin': ['Active Min'],
    'IdleMean': ['Idle Mean'],
    'IdleStd': ['Idle Std'],
    'IdleMax': ['Idle Max'],
    'IdleMin': ['Idle Min'],
}

LABEL_ALIASES = ['Classification', 'Label', 'label', 'Class', 'class']

TOKEN_NORMALIZATION = {
    'forward': 'fwd',
    'backward': 'bwd',
    'packet': 'pkt',
    'packets': 'pkt',
    'pkts': 'pkt',
    'length': 'len',
    'total': 'tot',
    'count': 'cnt',
    'counts': 'cnt',
    'average': 'avg',
    'variance': 'var',
    'segment': 'seg',
    'size': 'size',
    'bytes': 'byts',
    'byte': 'byts',
}

def normalize_name(name):
    return ''.join(ch.lower() for ch in str(name) if ch.isalnum())

def split_tokens(name):
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', str(name))
    text = re.sub(r'[^A-Za-z0-9]+', ' ', text)
    raw_tokens = [token.lower() for token in text.split() if token]
    return [TOKEN_NORMALIZATION.get(token, token) for token in raw_tokens]

def token_signature(name):
    return tuple(sorted(split_tokens(name)))

def find_best_column_match(columns, candidates):
    exact_map = {normalize_name(col): col for col in columns}
    signature_map = {}
    for col in columns:
        signature_map.setdefault(token_signature(col), col)

    for candidate in candidates:
        hit = exact_map.get(normalize_name(candidate))
        if hit:
            return hit

    for candidate in candidates:
        hit = signature_map.get(token_signature(candidate))
        if hit:
            return hit

    return None

def dataframe_mb(df):
    return float(df.memory_usage(deep=True).sum() / (1024 ** 2))

def has_csv_files(input_dir):
    return bool(glob.glob(os.path.join(input_dir, '**', '*.csv'), recursive=True))

def download_kaggle_dataset(dataset_slug, download_dir):
    os.makedirs(download_dir, exist_ok=True)
    if has_csv_files(download_dir):
        print('CSV files already exist in', download_dir, '- skip download')
        return
    completed = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', download_dir, '--unzip'],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        stderr = (completed.stderr or '').strip()
        stdout = (completed.stdout or '').strip()
        details = stderr or stdout or 'Kaggle CLI failed without output'
        raise RuntimeError(f'Could not download dataset {dataset_slug}: {details}')
    print('Downloaded dataset to', download_dir)

def resolve_required_columns(columns, label_override=''):
    resolved = {}
    for feature in APP_FEATURES:
        candidates = [feature] + FEATURE_ALIASES.get(feature, [])
        found = find_best_column_match(columns, candidates)
        if not found:
            raise KeyError(f"Missing required feature column for '{feature}'. Checked aliases: {candidates}")
        resolved[feature] = found

    if label_override:
        if label_override not in columns:
            raise KeyError(f"Could not find label column '{label_override}'")
        label_col = label_override
    else:
        label_col = find_best_column_match(columns, LABEL_ALIASES)
        if label_col is None:
            raise KeyError(f'Could not find label column. Tried: {LABEL_ALIASES}')
    return resolved, label_col

def sample_chunk(chunk, label_col, sample_frac):
    numeric_cols = [col for col in chunk.columns if col != label_col]
    chunk[numeric_cols] = chunk[numeric_cols].apply(pd.to_numeric, errors='coerce').astype('float32')
    chunk[label_col] = chunk[label_col].astype(str).str.strip()
    if sample_frac and 0 < sample_frac < 1:
        chunk = chunk.sample(frac=sample_frac, random_state=RANDOM_STATE)
    return chunk

def load_training_data(input_dir, label_override='', sample_frac=None, max_rows_per_file=None, chunk_size=50000):
    pattern = os.path.join(input_dir, '**', '*.csv')
    x_parts = []
    y_parts = []
    detected_label_col = None

    for path in sorted(glob.glob(pattern, recursive=True)):
        try:
            header = pd.read_csv(path, nrows=0)
            resolved_features, label_col = resolve_required_columns(header.columns.tolist(), label_override)
            usecols = list(dict.fromkeys(list(resolved_features.values()) + [label_col]))

            file_chunks = []
            for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunk_size, low_memory=False):
                chunk = sample_chunk(chunk, label_col, sample_frac)
                if not chunk.empty:
                    file_chunks.append(chunk)
                del chunk
                gc.collect()

            if not file_chunks:
                print('Skip:', path, 'no sampled rows kept from this file')
                continue

            file_df = pd.concat(file_chunks, ignore_index=True, copy=False)
            del file_chunks
            gc.collect()

            if max_rows_per_file and len(file_df) > max_rows_per_file:
                file_df = file_df.sample(n=max_rows_per_file, random_state=RANDOM_STATE)

            file_df = file_df.replace([np.inf, -np.inf], np.nan)
            selected = file_df[[resolved_features[f] for f in APP_FEATURES]].copy()
            selected.columns = APP_FEATURES
            selected = selected.astype('float32')
            labels = file_df[label_col].astype('category')

            x_parts.append(selected)
            y_parts.append(labels)
            detected_label_col = label_col

            print(f'Loaded: {path} rows={len(selected)} cols={len(selected.columns)} feature_mb={dataframe_mb(selected):.2f}')

            del header, file_df, selected, labels
            gc.collect()
        except Exception as exc:
            print('Skip:', path, exc)

    if not x_parts:
        raise ValueError('Could not find any CSV file for training')

    X = pd.concat(x_parts, ignore_index=True, copy=False)
    y = pd.concat(y_parts, ignore_index=True, copy=False).astype(str).str.strip()

    del x_parts, y_parts
    gc.collect()

    rare_counts = y.value_counts()
    rare_labels = rare_counts[rare_counts < 2].index.tolist()
    if rare_labels:
        mask = ~y.isin(rare_labels)
        X = X.loc[mask].reset_index(drop=True)
        y = y.loc[mask].reset_index(drop=True)
        print('Dropped rare classes with <2 rows after sampling:', rare_labels)

    return X, y, detected_label_col

def resolve_label_column(df_or_label, override=''):
    if override:
        return override
    if isinstance(df_or_label, str):
        return df_or_label
    label_col = find_best_column_match(list(df_or_label.columns), LABEL_ALIASES)
    if label_col:
        return label_col
    raise KeyError(f'Could not find label column. Tried: {LABEL_ALIASES}')

def build_autoencoder(input_dim):
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(64, activation='relu')(inputs)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dense(16, activation='relu')(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    outputs = keras.layers.Dense(input_dim, activation='linear')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return model

def save_class_distribution_plot(output_dir, y):
    counts = y.value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(counts.index.astype(str), counts.values, color='#2a6f97')
    ax.set_title('Dataset Class Distribution')
    ax.set_ylabel('Rows')
    ax.set_xlabel('Class')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'dataset_class_distribution.png'), dpi=160)
    plt.close(fig)

def save_classifier_plots(output_dir, y_test, y_pred, class_names, report):
    cm = confusion_matrix(y_test, y_pred, labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title('Random Forest Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    max_value = cm.max() if cm.size else 0
    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            cell = cm[row_idx, col_idx]
            color = 'white' if cell > (max_value / 2 if max_value else 0) else 'black'
            ax.text(col_idx, row_idx, str(cell), ha='center', va='center', color=color, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'classifier_confusion_matrix.png'), dpi=160)
    plt.close(fig)

    labels = []
    precision = []
    recall = []
    f1_score = []
    for class_name in class_names:
        if class_name not in report:
            continue
        labels.append(class_name)
        precision.append(report[class_name]['precision'])
        recall.append(report[class_name]['recall'])
        f1_score.append(report[class_name]['f1-score'])

    x_positions = np.arange(len(labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x_positions - width, precision, width=width, label='Precision', color='#4c956c')
    ax.bar(x_positions, recall, width=width, label='Recall', color='#f4a259')
    ax.bar(x_positions + width, f1_score, width=width, label='F1-score', color='#5b8e7d')
    ax.set_title('Random Forest Per-Class Scores')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'classifier_per_class_scores.png'), dpi=160)
    plt.close(fig)

def save_autoencoder_plots(output_dir, history, benign_mse, attack_mse):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.history.get('loss', []), label='Train Loss', color='#1d3557')
    ax.plot(history.history.get('val_loss', []), label='Validation Loss', color='#e76f51')
    ax.set_title('Autoencoder Training Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'autoencoder_loss_curve.png'), dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(benign_mse, bins=40, alpha=0.7, label='Benign', color='#457b9d')
    if len(attack_mse):
        ax.hist(attack_mse, bins=40, alpha=0.6, label='Attack', color='#d62828')
    ax.set_title('Autoencoder Reconstruction Error Distribution')
    ax.set_xlabel('Reconstruction MSE')
    ax.set_ylabel('Flow Count')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'autoencoder_reconstruction_error.png'), dpi=160)
    plt.close(fig)


## 4. T?i dataset Kaggle v? n?p d? li?u


In [ ]:
download_kaggle_dataset(DATASET_SLUG, DOWNLOAD_DIR)

X, y, detected_label_col = load_training_data(
    DOWNLOAD_DIR,
    label_override=LABEL_COLUMN_OVERRIDE,
    sample_frac=FILE_SAMPLE_FRAC,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    chunk_size=CHUNK_SIZE,
)
label_col = resolve_label_column(detected_label_col, LABEL_COLUMN_OVERRIDE)
gc.collect()

print('Rows:', len(X))
print('Features:', len(X.columns))
print('Label column:', label_col)
print('Sample fraction:', FILE_SAMPLE_FRAC)
print('Max rows per file:', MAX_ROWS_PER_FILE)
print('Approx X memory (MB):', round(dataframe_mb(X), 2))
print('Classes:', sorted(y.unique().tolist()))


## 5. Train Random Forest


In [ ]:
class_counts = y.value_counts()
stratify_labels = y if (class_counts >= 2).all() else None
if stratify_labels is None:
    print('Warning: some classes have <2 rows after sampling, train_test_split will run without stratify')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_labels,
)

del X, y
gc.collect()

rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=RF_ESTIMATORS,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ))
])

print('Train rows:', len(X_train))
print('Test rows:', len(X_test))
print('Training Random Forest...')
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
rf_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
print('RF accuracy:', rf_accuracy)


## 6. Train Autoencoder tr?n m?u benign


In [ ]:
X_benign_train = X_train.loc[y_train == BENIGN_LABEL]
X_benign_test = X_test.loc[y_test == BENIGN_LABEL]
X_attack_test = X_test.loc[y_test != BENIGN_LABEL]

if X_benign_train.empty:
    raise ValueError(f'Could not find benign train samples with label {BENIGN_LABEL}')
if X_benign_test.empty:
    raise ValueError(f'Could not find benign test samples with label {BENIGN_LABEL}')

if AE_MAX_BENIGN_TRAIN_ROWS and len(X_benign_train) > AE_MAX_BENIGN_TRAIN_ROWS:
    X_benign_train = X_benign_train.sample(n=AE_MAX_BENIGN_TRAIN_ROWS, random_state=RANDOM_STATE)
    print('Downsampled benign AE train rows to', len(X_benign_train))

ae_scaler = StandardScaler()
X_benign_train_scaled = ae_scaler.fit_transform(X_benign_train)
X_benign_test_scaled = ae_scaler.transform(X_benign_test)

X_b_train, X_b_val = train_test_split(
    X_benign_train_scaled,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

ae_model = build_autoencoder(X_benign_train_scaled.shape[1])
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]

print('Benign train rows:', len(X_benign_train))
print('Benign test rows:', len(X_benign_test))
print('Attack test rows:', len(X_attack_test))
print('Training Autoencoder...')
history = ae_model.fit(
    X_b_train,
    X_b_train,
    validation_data=(X_b_val, X_b_val),
    epochs=AE_EPOCHS,
    batch_size=AE_BATCH_SIZE,
    shuffle=True,
    verbose=1,
    callbacks=callbacks,
)

benign_recon = ae_model.predict(X_benign_test_scaled, verbose=0)
benign_mse = np.mean(np.square(benign_recon - X_benign_test_scaled), axis=1)
attack_mse = np.array([])

ae_metrics = {
    'benign_reconstruction_mse_mean': float(np.mean(benign_mse)),
    'benign_reconstruction_mse_std': float(np.std(benign_mse)),
}

if not X_attack_test.empty:
    X_attack_test_scaled = ae_scaler.transform(X_attack_test)
    attack_recon = ae_model.predict(X_attack_test_scaled, verbose=0)
    attack_mse = np.mean(np.square(attack_recon - X_attack_test_scaled), axis=1)
    ae_metrics['attack_reconstruction_mse_mean'] = float(np.mean(attack_mse))
    ae_metrics['attack_reconstruction_mse_std'] = float(np.std(attack_mse))

print(json.dumps(ae_metrics, indent=2))


## 7. T?o LIME, l?u model, metrics v? zip


In [ ]:
lime_sample = min(LIME_MAX_TRAIN_ROWS, len(X_train))
lime_frame = X_train.sample(n=lime_sample, random_state=RANDOM_STATE) if len(X_train) > lime_sample else X_train

print('Building LIME explainer...')
explainer = lime_tabular.LimeTabularExplainer(
    training_data=lime_frame.values,
    feature_names=APP_FEATURES,
    class_names=[str(label) for label in sorted(y_train.unique())],
    mode='classification',
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(rf, f)

joblib.dump(ae_scaler, os.path.join(OUTPUT_DIR, 'preprocess_pipeline_AE_39ft.save'))
ae_model.save(os.path.join(OUTPUT_DIR, 'autoencoder_39ft.hdf5'))

with open(os.path.join(OUTPUT_DIR, 'explainer'), 'wb') as f:
    dill.dump(explainer, f)

all_labels = pd.concat([y_train, y_test], ignore_index=True)
class_names = sorted(all_labels.astype(str).unique().tolist())
metrics = {
    'dataset_slug': DATASET_SLUG,
    'label_column': label_col,
    'benign_label': BENIGN_LABEL,
    'feature_order': APP_FEATURES,
    'class_names': class_names,
    'sample_fraction': FILE_SAMPLE_FRAC,
    'max_rows_per_file': MAX_ROWS_PER_FILE,
    'chunk_size': CHUNK_SIZE,
    'classifier': {
        'accuracy': rf_accuracy,
        'classification_report': rf_report,
    },
    'autoencoder': {
        **ae_metrics,
        'epochs_trained': len(history.history['loss']),
        'final_train_loss': float(history.history['loss'][-1]),
        'final_val_loss': float(history.history['val_loss'][-1]),
    },
}

with open(os.path.join(OUTPUT_DIR, 'training_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

report_lines = [
    '# Training Report',
    '',
    f"- Label column: `{label_col}`",
    f"- Benign label: `{BENIGN_LABEL}`",
    f"- Sample fraction: `{FILE_SAMPLE_FRAC}`",
    f"- Max rows per file: `{MAX_ROWS_PER_FILE}`",
    f"- Chunk size: `{CHUNK_SIZE}`",
    f"- Class names: {', '.join(class_names)}",
    f"- Random Forest accuracy: `{rf_accuracy:.4f}`",
    f"- Autoencoder epochs trained: `{len(history.history['loss'])}`",
    f"- Autoencoder final train loss: `{float(history.history['loss'][-1]):.6f}`",
    f"- Autoencoder final validation loss: `{float(history.history['val_loss'][-1]):.6f}`",
]

with open(os.path.join(OUTPUT_DIR, 'training_report.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

save_class_distribution_plot(OUTPUT_DIR, all_labels)
save_classifier_plots(OUTPUT_DIR, y_test, y_pred, class_names, rf_report)
save_autoencoder_plots(OUTPUT_DIR, history, benign_mse, attack_mse)

zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Saved artifacts to', OUTPUT_DIR)
print('Zip file:', zip_path)
print('Output files:', sorted(os.listdir(OUTPUT_DIR)))


## 8. Xem metrics t?m t?t


In [ ]:
metrics = json.loads(Path(OUTPUT_DIR, 'training_metrics.json').read_text(encoding='utf-8'))
print(json.dumps({
    'dataset_slug': metrics['dataset_slug'],
    'label_column': metrics['label_column'],
    'benign_label': metrics['benign_label'],
    'sample_fraction': metrics['sample_fraction'],
    'max_rows_per_file': metrics['max_rows_per_file'],
    'chunk_size': metrics['chunk_size'],
    'class_names': metrics['class_names'],
    'classifier_accuracy': metrics['classifier']['accuracy'],
    'autoencoder_epochs_trained': metrics['autoencoder']['epochs_trained'],
    'autoencoder_final_train_loss': metrics['autoencoder']['final_train_loss'],
    'autoencoder_final_val_loss': metrics['autoencoder']['final_val_loss'],
}, indent=2, ensure_ascii=False))


## 9. V? s? ?? k?t qu?


In [ ]:
plot_files = [
    'dataset_class_distribution.png',
    'classifier_confusion_matrix.png',
    'classifier_per_class_scores.png',
    'autoencoder_loss_curve.png',
    'autoencoder_reconstruction_error.png',
]

for plot_name in plot_files:
    img = mpimg.imread(Path(OUTPUT_DIR, plot_name))
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(plot_name)
    plt.show()


## 10. T?i file zip artifacts


In [ ]:
zip_path = Path(f'{OUTPUT_DIR}.zip')
assert zip_path.exists(), f'Could not find {zip_path}'
files.download(str(zip_path))
